In [1]:
!pip install python-dotenv langchain-community

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 3.2 MB/s eta 0:00:01
   -------- ------------------------------- 0.5/2.4 MB 3.2 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 1.5 MB/s eta 0:00:01
   ---------------------- ----------------- 1.3/2.4 MB 1.7 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 1.4 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 1.6 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 1.6 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 1.6 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 1.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ---------


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install neo4j


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
from langchain_community.graphs import Neo4jGraph
import warnings
warnings.filterwarnings("ignore")

C:\Users\madil\AppData\Local\Temp\ipykernel_43468\969532810.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import Neo4jGraph


In [3]:
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

OPENAI_ENDPOINT = os.getenv('OPENAI_BASE_URL') + '/embeddings'

In [4]:
kg = Neo4jGraph(url= NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)

In [5]:
kg.query("""
         CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS FOR
         (m:Movie) ON (m.taglineEmbedding)
         OPTIONS { indexConfig: {
             `vector.dimensions`: 1536,
             `vector.similarity_function`: 'cosine'
         }}
         """)

[]

In [6]:
kg.query("""SHOW VECTOR INDEXES""")

[{'id': 3,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [7]:
kg.query("MATCH (n) RETURN n LIMIT 5")

[{'n': {'taglineEmbedding': [0.0174501184374094,
    -0.005508840084075928,
    -0.002033882075920701,
    -0.025559445843100548,
    -0.01431905385106802,
    0.016703378409147263,
    -0.017109500244259834,
    0.0004974168259650469,
    -0.02520572766661644,
    -0.02952895499765873,
    0.0005616920534521341,
    0.020030954852700233,
    -0.006098370999097824,
    -0.004657295066863298,
    0.008102776482701302,
    -0.002950930269435048,
    0.02696121856570244,
    -0.03065561316907406,
    0.005725001450628042,
    -0.007840762846171856,
    -0.017463218420743942,
    0.016834385693073273,
    -0.006383311003446579,
    -0.03613170236349106,
    -0.011882325634360313,
    -0.010349544696509838,
    0.025022316724061966,
    -0.023214422166347504,
    0.01418804656714201,
    -0.022572487592697144,
    -0.0033603268675506115,
    -0.008580951951444149,
    0.010211987420916557,
    -0.024275578558444977,
    -0.004807953257113695,
    -0.010388846509158611,
    -0.01670337840914

In [8]:
kg.query("""
    MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL
    WITH movie, genai.vector.encode(
        movie.tagline, 
        "OpenAI", 
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS vector
    CALL db.create.setNodeVectorProperty(movie, "taglineEmbedding", vector)
    """, 
    params={"openAiApiKey":OPENAI_API_KEY, "openAiEndpoint": OPENAI_ENDPOINT} )

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.encode is deprecated. It is replaced by ai.text.embed.', position=<SummaryInputPosition line=3, column=17, offset=73>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 73, 'line': 3, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL\n    WITH movie, genai.vector.encode(\n        movie.tagline, \n        "OpenAI", \n        {\n          token: $openAiApiKey,\n          endpoint: $openAiEndpoint\n        }) AS vector\n    CALL db.create.setNodeVectorProperty(movie, "taglineEmbedding", vector)\n    '


[]

In [10]:
result = kg.query("""
                  MATCH (movie:Movie) WHERE movie.taglineEmbedding IS NOT NULL
                  return movie.tagline, movie.taglineEmbedding
                  limit 1
                  """)

In [11]:
result[0]['movie.tagline']

'Welcome to the Real World'

In [12]:
result[0]['movie.taglineEmbedding']

[0.017445066943764687,
 -0.005481892731040716,
 -0.002013522433117032,
 -0.025571243837475777,
 -0.014404304325580597,
 0.016737302765250206,
 -0.017078077420592308,
 0.000485358847072348,
 -0.025217361748218536,
 -0.029516370967030525,
 0.0005074764485470951,
 0.02000088058412075,
 -0.006091355811804533,
 -0.004649614915251732,
 0.008067196235060692,
 -0.002944100880995393,
 0.02686881087720394,
 -0.03064355067908764,
 0.005721090827137232,
 -0.007844381965696812,
 -0.017536813393235207,
 0.016855264082551003,
 -0.006379703991115093,
 -0.03604352846741676,
 -0.011894363909959793,
 -0.01031500194221735,
 0.02499454654753208,
 -0.023238245397806168,
 0.014220809563994408,
 -0.022635335102677345,
 -0.003411028301343322,
 -0.00857835914939642,
 0.010203594341874123,
 -0.024247463792562485,
 -0.004790512379258871,
 -0.010334662161767483,
 -0.01671108976006508,
 -0.01829700544476509,
 0.01129800733178854,
 0.00703176436945796,
 0.02838919311761856,
 -0.004253135994076729,
 0.006900697015225

In [13]:
len(result[0]['movie.taglineEmbedding'])

1536

In [14]:
question = "What movies are about Action?"

In [15]:
kg.query("""
    WITH genai.vector.encode(
        $question, 
        "OpenAI", 
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
    CALL db.index.vector.queryNodes(
        'movie_tagline_embeddings', 
        $top_k, 
        question_embedding
        ) YIELD node AS movie, score
    RETURN movie.title, movie.tagline, score
    """, 
    params={"openAiApiKey":OPENAI_API_KEY,
            "openAiEndpoint": OPENAI_ENDPOINT,
            "question": question,
            "top_k": 5
            })

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.encode is deprecated. It is replaced by ai.text.embed.', position=<SummaryInputPosition line=2, column=10, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 2, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    WITH genai.vector.encode(\n        $question, \n        "OpenAI", \n        {\n          token: $openAiApiKey,\n          endpoint: $openAiEndpoint\n        }) AS question_embedding\n    CALL db.index.vector.queryNodes(\n        \'movie_tagline_embeddings\', \n        $top_k, \n        question_embedding\n        ) YIELD node AS mo

[{'movie.title': 'RescueDawn',
  'movie.tagline': "Based on the extraordinary true story of one man's fight for freedom",
  'score': 0.8951952457427979},
 {'movie.title': 'Ninja Assassin',
  'movie.tagline': 'Prepare to enter a secret world of assassins',
  'score': 0.8873227834701538},
 {'movie.title': 'As Good as It Gets',
  'movie.tagline': 'A comedy from the heart that goes for the throat.',
  'score': 0.8868160247802734},
 {'movie.title': 'The Da Vinci Code',
  'movie.tagline': 'Break The Codes',
  'score': 0.8817266225814819},
 {'movie.title': 'Twister',
  'movie.tagline': "Don't Breathe. Don't Look Back.",
  'score': 0.8811315894126892}]